# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victorydavid-lab/Victory/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [27]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

print("DuckDB is ready.")

DuckDB is ready.


In [28]:
con = duckdb.connect()

print("DuckDB connection created.")

DuckDB connection created.


In [29]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

# Get Hugging Face token from Colab Secret
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

os.environ["HF_TOKEN"] = HF_TOKEN

# Connect DuckDB
con = duckdb.connect()

# Load HTTPFS for Hugging Face
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
}

print("FlyRank Week 4 connection ready.")
print(TABLES)

FlyRank Week 4 connection ready.
{'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"}


In [30]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

# ... all the Week 4 connection code ...

print("FlyRank Week 4 connection ready.")
print(TABLES)

FlyRank Week 4 connection ready.
{'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"}


In [31]:
con.execute(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT *
FROM {TABLES["fact_daily"]}
""")

print("fact_daily is ready.")

fact_daily is ready.


In [32]:
con.execute("""
SELECT COUNT(*) AS total_rows
FROM fact_daily
""").df()

,total_rows
0,9841378


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My modeling question is: can we identify client-content pairs that are likely to attract above-median organic traffic in the later part of March, using signals available before the decision date?

I will frame this as a binary classification problem. Client-content pairs with future organic sessions above the training-period median will be labelled as high organic traffic, while those at or below the median will be labelled as low organic traffic.

I will use Logistic Regression because it is appropriate for binary classification and provides an interpretable way to examine how observable content and traffic signals relate to the likelihood of higher future organic traffic. I will not assume that a more complex model is better; the model will be evaluated against the Week-4 baseline on a comparable evaluation set and metric.

The model will use only signals available up to the March 20 decision date. Organic sessions from March 21–31 will be used only to construct the held-out target and will not be used as model features.

This approach fits my Ranking Signal Analysis lane because it tests whether observable signals can help identify content with stronger subsequent organic traffic, without claiming that the model explains Google's ranking algorithm.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a time-aware split because the modeling question is about identifying content likely to attract higher organic traffic in a future period using information available at the decision moment.

I will use March 20 as the decision date. Features will be constructed only from information available up to March 20, while organic sessions from March 21-31 will be held out as the future target period.

The prediction unit will be a client-content pair rather than an individual daily row. This avoids treating repeated daily observations of the same content as independent examples.

The high-organic-traffic threshold will be calculated using the training/development data only and then applied to the held-out future period. Future organic sessions will be used only to construct the target and will not be included as model features.

I will not use a random split as the primary evaluation because randomly mixing dates could allow future information to enter the training data and would not represent the forward-looking use case. A random split may be used only as a secondary sensitivity check.

In [34]:
con.execute("""
SELECT
    month,
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM fact_daily
GROUP BY month
ORDER BY month
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,row_count,start_date,end_date
0,2026-03,9841378,2026-03-01,2026-03-31


In [35]:
con.execute("""
SELECT table_name
FROM information_schema.tables
ORDER BY table_name
""").df()

,table_name
0,fact_daily


In [36]:
con.execute("""
SELECT
    report_date,
    COUNT(*) AS row_count,
    COUNT(DISTINCT content_hash_id) AS unique_content,
    COUNT(DISTINCT client_hash_id) AS unique_clients
FROM fact_daily
GROUP BY report_date
ORDER BY report_date
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,row_count,unique_content,unique_clients
0,2026-03-01,275874,275874,51
1,2026-03-02,276269,276269,51
2,2026-03-03,311676,311676,52
3,2026-03-04,311675,311675,52
4,2026-03-05,311676,311676,52
5,2026-03-06,312187,312187,52
6,2026-03-07,312387,312387,52
7,2026-03-08,313374,313374,52
8,2026-03-09,313874,313874,52
9,2026-03-10,314047,314047,52


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Logistic Regression model outperformed the Week-4 baseline on ROC-AUC, Average Precision, Precision, and F1, while the baseline had slightly higher recall. The model therefore provides a stronger overall ranking signal for identifying content likely to attract future organic traffic, although it still misses some content that gains traffic without prior GSC visibility.

In [37]:
con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    sessions_organic
FROM fact_daily
WHERE report_date <= DATE '2026-03-20'
LIMIT 10
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>


In [38]:
con.execute("""
SELECT
    COUNT(*) AS total_future_rows,
    COUNT(sessions_organic) AS available_organic_rows,
    COUNT(*) - COUNT(sessions_organic) AS missing_organic_rows,
    ROUND(
        100.0 * COUNT(sessions_organic) / COUNT(*),
        2
    ) AS organic_available_pct
FROM fact_daily
WHERE report_date BETWEEN DATE '2026-03-21' AND DATE '2026-03-31'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_future_rows,available_organic_rows,missing_organic_rows,organic_available_pct
0,3591677,2823724,767953,78.62


In [39]:
con.execute("""
SELECT
    COUNT(*) AS training_rows,
    COUNT(*) FILTER (
        WHERE gsc_impressions IS NOT NULL
    ) AS impressions_available,
    COUNT(*) FILTER (
        WHERE gsc_avg_position IS NOT NULL
    ) AS position_available,
    COUNT(*) FILTER (
        WHERE sessions_organic IS NOT NULL
    ) AS organic_available
FROM fact_daily
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,training_rows,impressions_available,position_available,organic_available
0,6249701,6249701,2245946,3998913


In [40]:
con.execute("""
SELECT
    COUNT(*) AS unique_client_content_pairs
FROM (
    SELECT DISTINCT
        client_hash_id,
        content_hash_id
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20'
)
""").df()

,unique_client_content_pairs
0,323739


In [41]:
train_test_df = con.execute("""
WITH train_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position > 0 THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20'
    GROUP BY
        client_hash_id,
        content_hash_id
),

future_outcomes AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(sessions_organic) AS future_organic_sessions
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-21' AND DATE '2026-03-31'
      AND sessions_organic IS NOT NULL
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    t.*,
    f.future_organic_sessions
FROM train_features t
INNER JOIN future_outcomes f
    ON t.client_hash_id = f.client_hash_id
   AND t.content_hash_id = f.content_hash_id
""").df()

train_test_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,active_days,future_organic_sessions
0,client_9958f0a7ae1df715,content_810cf06597918291,281.0,1.0,8.771138,20,4.0
1,client_9958f0a7ae1df715,content_1d69c2ed06358f6f,60.0,0.0,12.711111,20,0.0
2,client_9958f0a7ae1df715,content_7483401e31fc5aa1,30.0,0.0,48.458333,20,0.0
3,client_9958f0a7ae1df715,content_a22ef2f4631595f1,299.0,0.0,33.876862,20,0.0
4,client_9958f0a7ae1df715,content_1ba03bc7001c2b84,19.0,0.0,54.637037,20,0.0


In [42]:
median_future_organic = train_test_df["future_organic_sessions"].median()

print("Training/development median future organic sessions:", median_future_organic)

train_test_df["target_high_organic"] = (
    train_test_df["future_organic_sessions"] > median_future_organic
).astype(int)

train_test_df["target_high_organic"].value_counts()

Training/development median future organic sessions: 0.0


,count
target_high_organic,
0,222084
1,32037


In [43]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

features = [
    "total_impressions",
    "total_clicks",
    "avg_position",
    "active_days"
]

X = train_test_df[features].copy()
y = train_test_df["target_high_organic"]

print("Features:", features)
print("Rows:", len(X))
print("High-organic rate:", round(y.mean(), 4))

Features: ['total_impressions', 'total_clicks', 'avg_position', 'active_days']
Rows: 254121
High-organic rate: 0.1261


In [44]:
pre_decision_median = con.execute("""
SELECT
    MEDIAN(sessions_organic) AS median_organic_sessions
FROM fact_daily
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-20'
  AND sessions_organic IS NOT NULL
""").fetchone()[0]

print("Pre-decision organic-session median:", pre_decision_median)

Pre-decision organic-session median: 0.0


In [45]:
con.execute("""
SELECT
    COUNT(*) AS total_pairs,
    COUNT(*) FILTER (WHERE future_organic_sessions > 0) AS organic_pairs,
    COUNT(*) FILTER (WHERE future_organic_sessions = 0) AS zero_organic_pairs,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE future_organic_sessions > 0)
        / COUNT(*),
        2
    ) AS organic_rate_pct
FROM train_test_df
""").df()

,total_pairs,organic_pairs,zero_organic_pairs,organic_rate_pct
0,254121,32037,222084,12.61


In [46]:
missing_check = train_test_df[[
    "total_impressions",
    "total_clicks",
    "avg_position",
    "active_days",
    "future_organic_sessions"
]].isna().sum()

print(missing_check)

total_impressions               0
total_clicks                    0
avg_position               137377
active_days                     0
future_organic_sessions         0
dtype: int64


In [47]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

features = [
    "total_impressions",
    "total_clicks",
    "avg_position",
    "active_days"
]

X = train_test_df[features].copy()
y = (train_test_df["future_organic_sessions"] > 0).astype(int)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X, y)

pred_prob = model.predict_proba(X)[:, 1]
pred = model.predict(X)

print("Model trained successfully.")
print("ROC-AUC:", round(roc_auc_score(y, pred_prob), 4))
print("Average Precision:", round(average_precision_score(y, pred_prob), 4))
print("Precision:", round(precision_score(y, pred), 4))
print("Recall:", round(recall_score(y, pred), 4))
print("F1:", round(f1_score(y, pred), 4))

Model trained successfully.
ROC-AUC: 0.9466
Average Precision: 0.7931
Precision: 0.561
Recall: 0.8394
F1: 0.6725


In [48]:
# Build a time-aware development/validation dataset.
# March 1–10 = model development
# March 11–20 = validation
# March 21–31 = future outcome

model_df = con.execute("""
WITH development_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position > 0 THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10'
    GROUP BY
        client_hash_id,
        content_hash_id
),

validation_outcomes AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(sessions_organic) AS validation_organic_sessions
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-11' AND DATE '2026-03-20'
      AND sessions_organic IS NOT NULL
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    d.*,
    v.validation_organic_sessions
FROM development_features d
INNER JOIN validation_outcomes v
    ON d.client_hash_id = v.client_hash_id
   AND d.content_hash_id = v.content_hash_id
""").df()

print("Rows:", len(model_df))
print(model_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 218570
            client_hash_id           content_hash_id  total_impressions  \
0  client_9958f0a7ae1df715  content_eb0aeedbcfaf2712              129.0   
1  client_9958f0a7ae1df715  content_108500096f9bc481              151.0   
2  client_9958f0a7ae1df715  content_4cec18f637b4c858               12.0   
3  client_9958f0a7ae1df715  content_a600692ebe905459               11.0   
4  client_9958f0a7ae1df715  content_3fe4eda1e378f05c               19.0   

   total_clicks  avg_position  active_days  validation_organic_sessions  
0           0.0     23.760453           10                          0.0  
1           2.0     11.980628           10                          0.0  
2           0.0     30.416667           10                          1.0  
3           0.0     69.633333           10                          0.0  
4           0.0     59.392857           10                          0.0  


In [49]:
model_df["target"] = (
    model_df["validation_organic_sessions"] > 0
).astype(int)

print("Validation target distribution:")
print(model_df["target"].value_counts())

print("\nPositive rate:")
print(round(model_df["target"].mean(), 4))

Validation target distribution:
target
0    195580
1     22990
Name: count, dtype: int64

Positive rate:
0.1052


In [50]:
# ============================================================
# WEEK 5 — FINAL TIME-AWARE MODEL + BASELINE COMPARISON
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. TRAINING DATA
# March 1–10 features
# March 11–20 organic traffic = training target
# ------------------------------------------------------------

train_df = con.execute("""
WITH features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-10'
    GROUP BY client_hash_id, content_hash_id
),

outcomes AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(sessions_organic) AS organic_sessions
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-11' AND DATE '2026-03-20'
      AND sessions_organic IS NOT NULL
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.*,
    o.organic_sessions
FROM features f
INNER JOIN outcomes o
    ON f.client_hash_id = o.client_hash_id
   AND f.content_hash_id = o.content_hash_id
""").df()


# ------------------------------------------------------------
# 2. FINAL TEST DATA
# March 11–20 features
# March 21–31 organic traffic = final test target
# ------------------------------------------------------------

test_df = con.execute("""
WITH features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-11' AND DATE '2026-03-20'
    GROUP BY client_hash_id, content_hash_id
),

outcomes AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(sessions_organic) AS organic_sessions
    FROM fact_daily
    WHERE report_date BETWEEN DATE '2026-03-21' AND DATE '2026-03-31'
      AND sessions_organic IS NOT NULL
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.*,
    o.organic_sessions
FROM features f
INNER JOIN outcomes o
    ON f.client_hash_id = o.client_hash_id
   AND f.content_hash_id = o.content_hash_id
""").df()


# ------------------------------------------------------------
# 3. CREATE BINARY TARGET
# Organic sessions > 0 = positive class
# ------------------------------------------------------------

train_df["target"] = (
    train_df["organic_sessions"] > 0
).astype(int)

test_df["target"] = (
    test_df["organic_sessions"] > 0
).astype(int)


# ------------------------------------------------------------
# 4. MODEL FEATURES
# ------------------------------------------------------------

features = [
    "total_impressions",
    "total_clicks",
    "avg_position",
    "active_days"
]

X_train = train_df[features]
y_train = train_df["target"]

X_test = test_df[features]
y_test = test_df["target"]


# ------------------------------------------------------------
# 5. TRAIN LOGISTIC REGRESSION
# ------------------------------------------------------------

model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    ("scaler", StandardScaler()),
    (
        "logistic",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.5).astype(int)


# ------------------------------------------------------------
# 6. WEEK-4-STYLE BASELINE
# High volume + poor position
#
# Thresholds are learned from TRAINING data only.
# ------------------------------------------------------------

volume_threshold = train_df["total_impressions"].quantile(0.80)
position_threshold = train_df["avg_position"].quantile(0.80)

test_df["baseline_score"] = (
    (test_df["total_impressions"] >= volume_threshold).astype(int) * 5
    +
    (test_df["avg_position"] >= position_threshold).astype(int) * 5
)

baseline_prob = test_df["baseline_score"] / 10


# ------------------------------------------------------------
# 7. EVALUATE BOTH ON THE SAME FINAL TEST SET
# ------------------------------------------------------------

model_roc_auc = roc_auc_score(y_test, model_prob)
model_ap = average_precision_score(y_test, model_prob)
model_precision = precision_score(y_test, model_pred, zero_division=0)
model_recall = recall_score(y_test, model_pred, zero_division=0)
model_f1 = f1_score(y_test, model_pred, zero_division=0)

baseline_roc_auc = roc_auc_score(y_test, baseline_prob)
baseline_ap = average_precision_score(y_test, baseline_prob)

baseline_pred = (baseline_prob >= 0.5).astype(int)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)


# ------------------------------------------------------------
# 8. MODEL VS BASELINE TABLE
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "ROC-AUC": [
        baseline_roc_auc,
        model_roc_auc
    ],
    "Average Precision": [
        baseline_ap,
        model_ap
    ],
    "Precision": [
        baseline_precision,
        model_precision
    ],
    "Recall": [
        baseline_recall,
        model_recall
    ],
    "F1": [
        baseline_f1,
        model_f1
    ]
})

print("TRAINING ROWS:", len(train_df))
print("FINAL TEST ROWS:", len(test_df))
print()
print("Training positive rate:", round(y_train.mean(), 4))
print("Final test positive rate:", round(y_test.mean(), 4))
print()
print("MODEL VS BASELINE")
display(comparison.round(4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TRAINING ROWS: 218570
FINAL TEST ROWS: 254121

Training positive rate: 0.1052
Final test positive rate: 0.1261

MODEL VS BASELINE


,method,ROC-AUC,Average Precision,Precision,Recall,F1
0,Week-4 baseline,0.8424,0.3457,0.3778,0.9166,0.5350
1,Logistic Regression,0.9460,0.7898,0.5066,0.8845,0.6442


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model's strongest positive signal was total clicks, followed by total impressions, indicating that existing search engagement and visibility were the strongest predictors of future organic traffic in this dataset. Average position had a relatively small negative coefficient, while missing position information had a stronger negative effect.

The main error pattern was false positives: the model identified 27,604 content items as likely to attract organic traffic that did not subsequently record organic sessions. However, there were only 3,699 false negatives. Several false negatives had zero GSC impressions and clicks but later generated substantial organic sessions, including examples with 99, 49, and 47 sessions. This suggests that the model is good at identifying opportunities where existing search signals are already visible, but it has limited ability to identify content that gains organic traffic without prior GSC visibility.

This means the model should be used as a prioritization signal rather than a definitive prediction. A useful improvement would be to incorporate additional signals that can capture emerging or previously invisible organic demand.

In [51]:
# ============================================================
# SECTION 4 — ERROR ANALYSIS
# ============================================================

error_df = test_df[[
    "client_hash_id",
    "content_hash_id",
    "total_impressions",
    "total_clicks",
    "avg_position",
    "active_days",
    "organic_sessions",
    "target"
]].copy()

error_df["predicted_probability"] = model_prob
error_df["prediction"] = model_pred

error_df["error_type"] = np.select(
    [
        (error_df["target"] == 1) & (error_df["prediction"] == 1),
        (error_df["target"] == 0) & (error_df["prediction"] == 0),
        (error_df["target"] == 0) & (error_df["prediction"] == 1),
        (error_df["target"] == 1) & (error_df["prediction"] == 0)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

print("Error distribution:")
print(error_df["error_type"].value_counts())

print("\nFalse positives:")
display(
    error_df[
        error_df["error_type"] == "False Positive"
    ]
    .sort_values("predicted_probability", ascending=False)
    .head(10)
)

print("\nFalse negatives:")
display(
    error_df[
        error_df["error_type"] == "False Negative"
    ]
    .sort_values("organic_sessions", ascending=False)
    .head(10)
)


Error distribution:
error_type
True Negative     194480
True Positive      28338
False Positive     27604
False Negative      3699
Name: count, dtype: int64

False positives:


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,active_days,organic_sessions,target,predicted_probability,prediction,error_type
235885,client_23a62021009f63c4,content_cbbb0c7db66e4333,13092.0,40.0,29.064394,10,0.0,0,1.0,1,False Positive
176544,client_20259bd6705d81d4,content_1f851827b37ec059,17476.0,34.0,7.723307,10,0.0,0,1.0,1,False Positive
182595,client_23a62021009f63c4,content_ee999e10a653e7b0,15099.0,70.0,10.692950,10,0.0,0,1.0,1,False Positive
182591,client_23a62021009f63c4,content_bd2aa238379a7497,8142.0,27.0,25.108047,10,0.0,0,1.0,1,False Positive
73138,client_e547b89c05043229,content_757b1fa67827358d,20659.0,6.0,7.178165,10,0.0,0,1.0,1,False Positive
247894,client_23a62021009f63c4,content_162c3ff050176978,10864.0,67.0,15.109544,10,0.0,0,1.0,1,False Positive
235362,client_23a62021009f63c4,content_ab91e088440ace78,23105.0,1.0,43.454009,10,0.0,0,1.0,1,False Positive
186361,client_23a62021009f63c4,content_92c7167740f1d3f5,1889.0,51.0,4.622719,10,0.0,0,1.0,1,False Positive
235422,client_23a62021009f63c4,content_559cdd76da9306de,29703.0,1.0,36.003735,10,0.0,0,1.0,1,False Positive
178272,client_23a62021009f63c4,content_164c1f53f13bcee1,30626.0,1.0,24.522436,10,0.0,0,1.0,1,False Positive



False negatives:


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,active_days,organic_sessions,target,predicted_probability,prediction,error_type
224225,client_b77d0d5f08f05e64,content_0b90a011c880d1f5,0.0,0.0,NaN,10,99.0,1,0.063006,0,False Negative
213195,client_73cda7b4e4f265ea,content_b04316f952e1f015,0.0,0.0,NaN,7,49.0,1,0.201150,0,False Negative
197463,client_73cda7b4e4f265ea,content_2c1c3a436eb74359,0.0,0.0,NaN,7,47.0,1,0.201150,0,False Negative
197456,client_73cda7b4e4f265ea,content_eb6ab6c1ab67601c,0.0,0.0,NaN,7,39.0,1,0.201150,0,False Negative
250845,client_3f0ce4d44fe94f3d,content_5ba04992417e0c4a,0.0,0.0,NaN,10,34.0,1,0.063006,0,False Negative
197464,client_73cda7b4e4f265ea,content_ff0bf6237ad4cd8d,0.0,0.0,NaN,7,32.0,1,0.201150,0,False Negative
212335,client_86ebc2f12c01f586,content_31ca88682b5b4b73,0.0,0.0,NaN,10,31.0,1,0.063006,0,False Negative
34474,client_fef1a8f436438636,content_fa6ddfda48399e4b,89.0,0.0,24.532619,10,31.0,1,0.413657,0,False Negative
213192,client_73cda7b4e4f265ea,content_06c43bb6c9c840e1,0.0,0.0,NaN,7,30.0,1,0.201150,0,False Negative
197459,client_73cda7b4e4f265ea,content_09e15c07e74055ee,0.0,0.0,NaN,7,27.0,1,0.201150,0,False Negative


In [52]:
# ============================================================
# FEATURE INTERPRETATION
# ============================================================

logistic_model = model.named_steps["logistic"]

# Get feature names after the imputer adds missing-value indicators
imputer = model.named_steps["imputer"]

feature_names = imputer.get_feature_names_out(features)

coefficients = logistic_model.coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_effect": np.abs(coefficients)
}).sort_values(
    "absolute_effect",
    ascending=False
)

print("Logistic Regression feature effects:")
display(feature_importance)

Logistic Regression feature effects:


,feature,coefficient,absolute_effect
1,total_clicks,4.597451,4.597451
0,total_impressions,2.029234,2.029234
4,missingindicator_avg_position,-1.163249,1.163249
3,active_days,-0.437500,0.437500
2,avg_position,-0.149377,0.149377


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.